In [1]:
import pandas as pd
from tqdm import tqdm
from wandb.apis.public import Run

import wandb

In [2]:
%cd "/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/"

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [3]:
# --- Setup ---

CSV_PATH = "test_grid_experiments_extended.csv"
WANDB_ENTITY = "roesch01-university-of-mannheim"  # Falls nötig, hier deinen Usernamen/Teamnamen eintragen
WANDB_PROJECT = "master-thesis-extended"


METRICS = {
    "eval/recall_concepts": "Recall Concepts",
    "eval/precision_concepts": "Precision Concepts",
    "eval/total": "Total Loss",
    "eval/loss_concepts": "Concept Loss",
    "eval/iou_mean": "Mean IoU",
    "eval/foreground_dice_scores": "Foreground Dice",
    "eval/f1_concept_activations": r"$F_1$-Score Concepts",
    "eval/f1_labels": r"$F_1$-Score Labels",
    "eval/dice_mean": "Mean Dice",
    "eval/accuracy_concepts": "Concept Accuracy",
    "eval/accuracy_labels": "Label Accuracy",
    "_runtime": "Runtime",
}

In [4]:
def get_data():
    
    df_meta = pd.read_csv(CSV_PATH, sep=';', index_col=None)
    api = wandb.Api()
    all_history: list[pd.DataFrame] = []

    for index, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
        if pd.isna(row['wandb_run_id']): 
            continue

        try:
            run: Run = api.run(f"{WANDB_PROJECT}/{row['wandb_run_id']}")

            available_keys = set(run.summary.keys())

            # Schnittmenge mit deinen gewünschten Metriken
            valid_keys = [k for k in METRICS.keys() if k in available_keys]

            hist: pd.DataFrame = run.history(keys=list(valid_keys)) # type: ignore

            if hist.empty: 
                print("empty")
                continue

            # --- EPOCH NORMALISIERUNG ---
            # Wir ignorieren den echten Step und zählen einfach hoch (1, 2, 3...)
            hist = hist.sort_index().reset_index(drop=True)
            hist['normalized_epoch'] = hist.index + 1
            hist['epoch'] = hist.index

            for tag in run.tags:
                hist[f'tag_{tag}'] = True

            # Metadaten
            hist['run_id'] = row['wandb_run_id']
            hist['concept_module'] = row['concept_module_name']
            hist['segmentation_module'] = row['segmentation_module_name']
            hist['concept_criterion'] = row['concept_criterion']
            hist['use_soft_labels'] = row['use_soft_labels']
            hist['unified_model'] = row['unified_name'] if row['unified_name'] != "nan" else None

            hist['segmentation_module'] = hist['unified_model'] if hist['unified_model'].notna().any() else hist['segmentation_module']
            
            hist['dataset'] = row['dataset']
            
            all_history.append(hist)
        except Exception as e:
            print(f"Fehler bei Run {row['wandb_run_id']}: {e}")

    full_df = pd.concat(all_history, ignore_index=True)
    return full_df

In [5]:
data = get_data()

100%|██████████| 397/397 [01:12<00:00,  5.49it/s]


In [6]:
segmentation_mapping = {
        "segdino_b": "SegDINO",
            "SegmentationHeadSETRPUP": "SETR-PUP",
                "SegmentationHeadUpscaledSingle": "Upscaled SingleLayer",
                    "SegmentationHeadUpscaledMulti": "Upscaled MultiLayer",
                    }

concept_mapping = {
    "SegMaskAvgPool": "Global Average Pooling",
        "SegMaskAvgPoolTrainChannelAffine": "Affine-Calibrated Global Average Pooling",
            "LogitMeanTopK": r"Top-$\rho$ Average Pooling",
                "SegMaskMaxPool": "Maximum Pooling",
                    "SALFConceptModule": "Spatial Softmax Attention Pooling",
                    }

use_soft_labels_mapping = {
    'nan': "no",
    '--use-soft-labels': 'yes',
}


data["concept_module"] = data["concept_module"].replace(concept_mapping)
data["segmentation_module"] = data["segmentation_module"].replace(segmentation_mapping)
data["use_soft_labels"] = data["use_soft_labels"].astype(str).replace(use_soft_labels_mapping)

In [7]:
data.head()

,_step,eval/recall_concepts,eval/precision_concepts,eval/total,eval/iou_mean,eval/foreground_dice_scores,eval/f1_concept_activations,eval/f1_labels,eval/dice_mean,eval/accuracy_concepts,...,unified_model,dataset,eval/loss_concepts,tag_aab-con,tag_aab-con-maskreg-intro,tag_aab-soft-bce,tag_aab-aff,tag_joint,tag_con-implicit-es,tag_con-implicit-cs
0,781,0.038462,0.001795,0.289903,0.719944,0.452890,0.00343,0.0,0.734870,0.779231,...,NaN,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1562,0.000000,0.000000,0.271712,0.769979,0.479904,0.00000,0.0,0.785506,0.814103,...,NaN,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2343,0.000000,0.000000,0.264517,0.790440,0.488555,0.00000,0.0,0.806390,0.814103,...,NaN,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3124,0.000000,0.000000,0.259402,0.794077,0.499254,0.00000,0.0,0.810149,0.814103,...,NaN,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3906,0.000000,0.000000,0.254300,0.799025,0.505767,0.00000,0.0,0.814970,0.814103,...,NaN,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
data_renamed = data.rename(columns={
    "concept_module": "Concept Module",
    "segmentation_module": "Segmentation Module",
    "concept_criterion": "Concept Criterion",
    "use_soft_labels": "Use Soft Labels",
    "dataset": "Dataset",
    **METRICS
})

data_renamed["Concept Module"] = pd.Categorical(
    data_renamed["Concept Module"],
    categories=list(concept_mapping.values()),
    ordered=True
)

# 2️⃣ Segmentation Module ebenfalls
data_renamed["Segmentation Module"] = pd.Categorical(
    data_renamed["Segmentation Module"],
    categories=list(segmentation_mapping.values()),
    ordered=True
)

# 3️⃣ Sortieren
df_sorted = data_renamed.sort_values(
    by=["Concept Module", "Segmentation Module"],
    ascending=True
)

In [9]:
df_sorted.head()

,_step,Recall Concepts,Precision Concepts,Total Loss,Mean IoU,Foreground Dice,$F_1$-Score Concepts,$F_1$-Score Labels,Mean Dice,Concept Accuracy,...,unified_model,Dataset,Concept Loss,tag_aab-con,tag_aab-con-maskreg-intro,tag_aab-soft-bce,tag_aab-aff,tag_joint,tag_con-implicit-es,tag_con-implicit-cs
69,781,0.0,0.0,0.244985,0.783426,0.526457,0.0,0.006667,0.808431,0.814103,...,segdino_b,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,1562,0.0,0.0,0.222595,0.804496,0.566505,0.0,0.006667,0.830091,0.814103,...,segdino_b,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,2343,0.0,0.0,0.214856,0.809113,0.581208,0.0,0.006667,0.834639,0.814103,...,segdino_b,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,3124,0.0,0.0,0.207240,0.836924,0.592973,0.0,0.006667,0.862481,0.814103,...,segdino_b,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,3906,0.0,0.0,0.203963,0.832565,0.599443,0.0,0.006667,0.858034,0.814103,...,segdino_b,FunnyBirds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
assert (df_sorted['tag_Final'] != True).sum() == 0

In [11]:
df_sorted['run_id'].nunique()

66

In [12]:
df_sorted.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

In [13]:
df_sorted.to_parquet("thesis-figures/extended_cbm/results/1_extended_cbm_results.parquet")